<a href="https://colab.research.google.com/github/shivainlabs/Introduction-to-Deep-Learning-and-GenAI/blob/main/Week%2011/RAG_Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# models are trained on data available publically
"""
question -> BOX1 (upload student handbook,
Extract relevent data using BOX1, then pass LLM)
 -> LLM -> Answer


** BOX1 Works **

Documents -> Load Doc -> Document parsing
-> chunking -> Embedding Model -> Vector representation
-> vector database (Eg. FAISS) ->

"""

'\nquestion -> BOX1 (upload student handbook,\nExtract relevent data using BOX1, then pass LLM)\n -> LLM -> Answer\n\n\n** BOX1 Works **\n\nDocuments -> Load Doc -> Document parsing\n-> chunking -> Embedding Model -> Vector representation\n-> vector database (Eg. FAISS) ->\n\n'

In [ ]:
"""
** Vector database **
Useful to query relevent information and indexing


user question -> Embedding model -> vector representation
-> similarity search (in FAISS) -> Retrieve relevent chunks + prompt (could be question)
-> LLM -> Answer

"""

'\n** Vector database **\nUseful to query relevent information and indexing\n\n\nuser question -> Embedding model -> vector representation\n-> similarity search (in FAISS) -> Retrieve relevent chunks + prompt (could be question)\n-> LLM -> Answer\n\n'

In [ ]:
!pip install -q \
    langchain \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-google-genai \
    langchain-groq \
    faiss-cpu \
    pypdf

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # For chunking the data properly
from langchain_google_genai import GoogleGenerativeAIEmbeddings # Embedding model
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_18232/225921310.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader


In [ ]:
import os
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print('Groq key set: ', bool(os.environ.get("GROQ_API_KEY")))

Groq key set:  True


In [ ]:
# Document(page_content="actual text...",metadata={"source":"--"})

# we convert our data to Document then, it would become easier to process with Langchain ...



In [ ]:
knowledge = """ZephyrBot 3000 — Owner's Guide (excerpt)

Overview
The ZephyrBot 3000 is an indoor home-assistant robot manufactured by Zephyr Robotics,
first released in March 2027. It stands 42 centimeters tall and weighs 3.1 kilograms.

Battery and charging
The ZephyrBot 3000 runs on a removable 5200 mAh lithium-ion battery. On a full charge it
operates for about 6 hours of active use. It fully recharges in 90 minutes using the
included 45-watt dock. The battery is rated for 800 full charge cycles.

Connectivity
The robot connects over Wi-Fi 6 and Bluetooth 5.3. It does NOT support Ethernet.
Voice commands are processed locally on-device; no audio is sent to the cloud.

Warranty
Every ZephyrBot 3000 includes a 2-year limited warranty covering manufacturing defects.
The warranty does not cover water damage or damage from unauthorized repairs.

Support
For support, contact help@zephyr-robotics.example between 9am and 6pm Pacific Time.
The default administrator password on a new unit is "zephyr-admin" and should be
changed during first-time setup.
"""

with open("zephyrbot_manual.txt", "w") as f:
    f.write(knowledge)

print("Wrote zephyrbot_manual.txt")

Wrote zephyrbot_manual.txt


In [ ]:
loader = TextLoader("zephyrbot_manual.txt")
document = loader.load() # list where each element of this list is Document datatype
document

[Document(metadata={'source': 'zephyrbot_manual.txt'}, page_content='ZephyrBot 3000 — Owner\'s Guide (excerpt)\n\nOverview\nThe ZephyrBot 3000 is an indoor home-assistant robot manufactured by Zephyr Robotics,\nfirst released in March 2027. It stands 42 centimeters tall and weighs 3.1 kilograms.\n\nBattery and charging\nThe ZephyrBot 3000 runs on a removable 5200 mAh lithium-ion battery. On a full charge it\noperates for about 6 hours of active use. It fully recharges in 90 minutes using the\nincluded 45-watt dock. The battery is rated for 800 full charge cycles.\n\nConnectivity\nThe robot connects over Wi-Fi 6 and Bluetooth 5.3. It does NOT support Ethernet.\nVoice commands are processed locally on-device; no audio is sent to the cloud.\n\nWarranty\nEvery ZephyrBot 3000 includes a 2-year limited warranty covering manufacturing defects.\nThe warranty does not cover water damage or damage from unauthorized repairs.\n\nSupport\nFor support, contact help@zephyr-robotics.example between 9a

In [ ]:
document[0].metadata

{'source': 'zephyrbot_manual.txt'}

In [ ]:
document[0].page_content

'ZephyrBot 3000 — Owner\'s Guide (excerpt)\n\nOverview\nThe ZephyrBot 3000 is an indoor home-assistant robot manufactured by Zephyr Robotics,\nfirst released in March 2027. It stands 42 centimeters tall and weighs 3.1 kilograms.\n\nBattery and charging\nThe ZephyrBot 3000 runs on a removable 5200 mAh lithium-ion battery. On a full charge it\noperates for about 6 hours of active use. It fully recharges in 90 minutes using the\nincluded 45-watt dock. The battery is rated for 800 full charge cycles.\n\nConnectivity\nThe robot connects over Wi-Fi 6 and Bluetooth 5.3. It does NOT support Ethernet.\nVoice commands are processed locally on-device; no audio is sent to the cloud.\n\nWarranty\nEvery ZephyrBot 3000 includes a 2-year limited warranty covering manufacturing defects.\nThe warranty does not cover water damage or damage from unauthorized repairs.\n\nSupport\nFor support, contact help@zephyr-robotics.example between 9am and 6pm Pacific Time.\nThe default administrator password on a new

## Chunking

In [ ]:
"""

Chunks too small -> Not enough information

Chunks too large -> Each chunk mixes many topics
(Extra irrelvent information will reduce the score)

Overlap : So, that we can avoid sudden stop boundary

"""

'\n\nChunks too small -> Not enough information\n\nChunks too large -> Each chunk mixes many topics\n(Extra irrelvent information will reduce the score)\n\nOverlap : So, that we can avoid sudden stop boundary\n\n'

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 50,
)

chunks = splitter.split_documents(document)

In [ ]:
chunks

[Document(metadata={'source': 'zephyrbot_manual.txt'}, page_content="ZephyrBot 3000 — Owner's Guide (excerpt)"),
 Document(metadata={'source': 'zephyrbot_manual.txt'}, page_content='Overview\nThe ZephyrBot 3000 is an indoor home-assistant robot manufactured by Zephyr Robotics,\nfirst released in March 2027. It stands 42 centimeters tall and weighs 3.1 kilograms.'),
 Document(metadata={'source': 'zephyrbot_manual.txt'}, page_content='Battery and charging\nThe ZephyrBot 3000 runs on a removable 5200 mAh lithium-ion battery. On a full charge it\noperates for about 6 hours of active use. It fully recharges in 90 minutes using the'),
 Document(metadata={'source': 'zephyrbot_manual.txt'}, page_content='included 45-watt dock. The battery is rated for 800 full charge cycles.'),
 Document(metadata={'source': 'zephyrbot_manual.txt'}, page_content='Connectivity\nThe robot connects over Wi-Fi 6 and Bluetooth 5.3. It does NOT support Ethernet.\nVoice commands are processed locally on-device; no aud

In [ ]:
for i,c in enumerate(chunks):
  print(f"chunk {i}: {c.page_content}\n")

chunk 0: ZephyrBot 3000 — Owner's Guide (excerpt)

chunk 1: Overview
The ZephyrBot 3000 is an indoor home-assistant robot manufactured by Zephyr Robotics,
first released in March 2027. It stands 42 centimeters tall and weighs 3.1 kilograms.

chunk 2: Battery and charging
The ZephyrBot 3000 runs on a removable 5200 mAh lithium-ion battery. On a full charge it
operates for about 6 hours of active use. It fully recharges in 90 minutes using the

chunk 3: included 45-watt dock. The battery is rated for 800 full charge cycles.

chunk 4: Connectivity
The robot connects over Wi-Fi 6 and Bluetooth 5.3. It does NOT support Ethernet.
Voice commands are processed locally on-device; no audio is sent to the cloud.

chunk 5: Warranty
Every ZephyrBot 3000 includes a 2-year limited warranty covering manufacturing defects.
The warranty does not cover water damage or damage from unauthorized repairs.

chunk 6: Support
For support, contact help@zephyr-robotics.example between 9am and 6pm Pacific Time.
Th

In [ ]:
import os
from google.colab import userdata

os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
print('Google key set: ', bool(os.environ.get("GOOGLE_API_KEY")))

Google key set:  True


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model = 'gemini-embedding-001',
    output_dimensionality = 768
)

In [ ]:
vector_store = FAISS.from_documents(chunks, embeddings) # 8 X 768 (Matrix)
print(vector_store.index.ntotal) # To see, how many chunks
print(vector_store.index.d) # Dimension

8
768


In [ ]:
question = "How long does the battery last on a full charge"
results = vector_store.similarity_search(question,k=3) # Find the similiary score, give top 3 chunks
results

[Document(id='c2b24211-dc61-475b-8d14-4c102b7c5f0f', metadata={'source': 'zephyrbot_manual.txt'}, page_content='Battery and charging\nThe ZephyrBot 3000 runs on a removable 5200 mAh lithium-ion battery. On a full charge it\noperates for about 6 hours of active use. It fully recharges in 90 minutes using the'),
 Document(id='f0639e8c-7d9e-4fc2-8155-422ebbba41d1', metadata={'source': 'zephyrbot_manual.txt'}, page_content='included 45-watt dock. The battery is rated for 800 full charge cycles.'),
 Document(id='30db0b7b-3878-45b2-aa60-2fb1f8050a09', metadata={'source': 'zephyrbot_manual.txt'}, page_content='Support\nFor support, contact help@zephyr-robotics.example between 9am and 6pm Pacific Time.\nThe default administrator password on a new unit is "zephyr-admin" and should be')]

In [ ]:
for i,res in enumerate(results):
  print(f"{i} - {res.page_content}\n")

0 - Battery and charging
The ZephyrBot 3000 runs on a removable 5200 mAh lithium-ion battery. On a full charge it
operates for about 6 hours of active use. It fully recharges in 90 minutes using the

1 - included 45-watt dock. The battery is rated for 800 full charge cycles.

2 - Support
For support, contact help@zephyr-robotics.example between 9am and 6pm Pacific Time.
The default administrator password on a new unit is "zephyr-admin" and should be



In [ ]:
score = vector_store.similarity_search_with_score(question,k=3) # with score
score

[(Document(id='c2b24211-dc61-475b-8d14-4c102b7c5f0f', metadata={'source': 'zephyrbot_manual.txt'}, page_content='Battery and charging\nThe ZephyrBot 3000 runs on a removable 5200 mAh lithium-ion battery. On a full charge it\noperates for about 6 hours of active use. It fully recharges in 90 minutes using the'),
  np.float32(0.20621574)),
 (Document(id='f0639e8c-7d9e-4fc2-8155-422ebbba41d1', metadata={'source': 'zephyrbot_manual.txt'}, page_content='included 45-watt dock. The battery is rated for 800 full charge cycles.'),
  np.float32(0.21554491)),
 (Document(id='30db0b7b-3878-45b2-aa60-2fb1f8050a09', metadata={'source': 'zephyrbot_manual.txt'}, page_content='Support\nFor support, contact help@zephyr-robotics.example between 9am and 6pm Pacific Time.\nThe default administrator password on a new unit is "zephyr-admin" and should be'),
  np.float32(0.27371112))]

In [ ]:
template = """You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know based on the provided documents."
Keep the answer concise and factual.

Context:
{context}

Question:
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)


In [ ]:
llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
)

tester = llm.invoke('Explain RAG in one sentence')

In [ ]:
tester.content

'RAG (Retrieval‑Augmented Generation) is a hybrid NLP technique that first pulls relevant documents from a knowledge base and then feeds them into a language model to generate more accurate, context‑grounded responses.'

In [ ]:
tester2 = llm.invoke("what is the rating of the battery of ZephyrBot")
tester2.content

'I’m not familiar with a specific product called “ZephyrBot,” so I don’t have the exact battery rating on hand. If you can point me to a brand, model number, or a source (like a product page or spec sheet), I’d be happy to look up the details for you.'

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):

    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
    {"context":retriever | format_docs, "question": RunnablePassthrough() }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
answer = rag_chain.invoke("How long does the battery last on a full charge")
print(answer)

About 6 hours of active use on a full charge.
